In [4]:

# Crypto Lab Assignment 9
## Evaluation of Operating-System Random Number Generators

### Objective
Generate random data using Linux `/dev/urandom`, OpenSSL's
`rand` utility, and Windows CNG/.NET cryptographic APIs.

Analyze byte frequencies, bit distributions, repeated patterns,
serial correlations, and statistical tests.

### Important
Statistical tests evaluate observable properties of the generated
data. Passing them does not prove unpredictability or cryptographic
security.
  

SyntaxError: unterminated string literal (detected at line 5) (3802129418.py, line 5)

In [ ]:

from pathlib import Path
import math
import numpy as np
from scipy.stats import chisquare, binomtest, norm

DATA_DIR = Path("outputs/data")
files = sorted(DATA_DIR.glob("*.bin"))

if not files:
    raise FileNotFoundError(
        "No .bin files found in outputs/data. Generate the data first."
    )

for file in files:
    print(file.name, file.stat().st_size, "bytes")
  

In [ ]:

def byte_frequency_test(data):
    counts = np.bincount(
        np.frombuffer(data, dtype=np.uint8),
        minlength=256
    )

    expected = np.full(256, len(data) / 256)
    statistic, p_value = chisquare(counts, f_exp=expected)

    return counts, statistic, p_value


def bit_balance_test(data):
    bits = np.unpackbits(
        np.frombuffer(data, dtype=np.uint8)
    )

    ones = int(bits.sum())
    total = len(bits)

    result = binomtest(ones, total, 0.5)

    return ones, total - ones, result.pvalue


for file in files:
    data = file.read_bytes()

    counts, chi_stat, chi_p = byte_frequency_test(data)
    ones, zeros, bit_p = bit_balance_test(data)

    print("\n", file.name)
    print("Bytes:", len(data))
    print("Byte chi-square statistic:", chi_stat)
    print("Byte chi-square p-value:", chi_p)
    print("Zero bits:", zeros)
    print("One bits:", ones)
    print("Bit-balance p-value:", bit_p)
  

In [ ]:

def runs_test(data):
    bits = np.unpackbits(
        np.frombuffer(data, dtype=np.uint8)
    ).astype(np.int8)

    n0 = int(np.sum(bits == 0))
    n1 = int(np.sum(bits == 1))

    if n0 == 0 or n1 == 0:
        return None, None

    runs = 1 + int(np.count_nonzero(bits[1:] != bits[:-1]))

    expected = 1 + (2 * n0 * n1) / (n0 + n1)

    variance = (
        2 * n0 * n1 * (2 * n0 * n1 - n0 - n1)
        / (((n0 + n1) ** 2) * (n0 + n1 - 1))
    )

    if variance <= 0:
        return runs, None

    z = (runs - expected) / math.sqrt(variance)
    p_value = 2 * norm.sf(abs(z))

    return runs, p_value


def serial_correlation(data):
    values = np.frombuffer(data, dtype=np.uint8).astype(np.float64)

    if len(values) < 2 or np.var(values) == 0:
        return None

    return float(np.corrcoef(values[:-1], values[1:])[0, 1])


for file in files:
    data = file.read_bytes()

    runs, runs_p = runs_test(data)

    print("\n", file.name)
    print("Number of bit runs:", runs)
    print("Runs-test p-value:", runs_p)
    print("Serial correlation:", serial_correlation(data))
  

In [ ]:

from collections import Counter
import pandas as pd


def repeated_pattern_analysis(data, block_size=4):
    usable = len(data) - len(data) % block_size
    blocks = [
        data[i:i + block_size]
        for i in range(0, usable, block_size)
    ]

    counts = Counter(blocks)

    return {
        "blocks_checked": len(blocks),
        "unique_blocks": len(counts),
        "repeated_block_types": sum(
            count > 1 for count in counts.values()
        ),
        "extra_duplicate_occurrences": sum(
            count - 1 for count in counts.values() if count > 1
        )
    }


rows = []

for file in files:
    data = file.read_bytes()

    _, byte_chi, byte_p = byte_frequency_test(data)
    ones, zeros, bit_p = bit_balance_test(data)
    runs, runs_p = runs_test(data)

    rows.append({
        "File": file.name,
        "Bytes": len(data),
        "Byte chi-square": byte_chi,
        "Byte p-value": byte_p,
        "Bit p-value": bit_p,
        "Runs": runs,
        "Runs p-value": runs_p,
        "Serial correlation": serial_correlation(data),
        "Repeated patterns": repeated_pattern_analysis(data)
    })

results = pd.DataFrame(rows)
display(results)
  

In [ ]:

## Interpretation

1. **Byte-frequency chi-square test:** Compares the 256 observed
byte counts with the expected equal counts. A small p-value may
indicate a deviation from the uniform-byte model.

2. **Bit-balance test:** Checks whether the number of one-bits is
consistent with a probability of 0.5.

3. **Runs test:** Checks whether the sequence of zero-bits and
one-bits has a plausible number of consecutive runs.

4. **Serial correlation:** Measures linear dependence between
adjacent byte values. A value close to zero indicates little
linear correlation at lag one.

5. **Repeated patterns:** Counts repeated non-overlapping
four-byte blocks. Some repetitions can occur by chance.

A p-value above 0.05 means the test did not find statistically
significant evidence against its particular null hypothesis at
the 5% significance level. It does not prove that the data are
random or cryptographically secure.

The tests examine different properties and do not guarantee
independence, unpredictability, correct entropy management, or
resistance to an attacker.
  